# EXPERIMENT 2B – Diabetes Dataset Statistical and Regression Analysis
## Aim
Perform univariate, bivariate, logistic and multiple regression analyses on UCI Diabetes and Pima Indians Diabetes datasets, and compare them.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, classification_report

## 2. Load datasets

In [ ]:
# Pima Indians Diabetes: CSV hosted in the public Brownlee dataset repository.
pima_url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
pima_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin",
             "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
pima = pd.read_csv(pima_url, names=pima_cols)

# UCI Diabetes dataset available through scikit-learn.
# It has a continuous disease-progression target, unlike Pima's binary Outcome.
from sklearn.datasets import load_diabetes
uci_raw = load_diabetes()
uci = pd.DataFrame(uci_raw.data, columns=uci_raw.feature_names)
uci["DiseaseProgression"] = uci_raw.target

print("Pima shape:", pima.shape, "| UCI shape:", uci.shape)
display(pima.head())
display(uci.head())

## 3. Preprocess datasets

In [ ]:
# In Pima, zero in these medical measurements commonly represents missing data.
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
pima[zero_cols] = pima[zero_cols].replace(0, np.nan)
# Fill missing measurements with each column's median.
pima[zero_cols] = pima[zero_cols].fillna(pima[zero_cols].median())

pima.drop_duplicates(inplace=True)
uci.drop_duplicates(inplace=True)
print("Pima missing values:\n", pima.isnull().sum())
print("UCI missing values:\n", uci.isnull().sum())

## 4. Univariate analysis

In [ ]:
# Calculate frequency, mean, median, mode, variance, standard deviation,
# skewness and kurtosis for each numeric variable.
def univariate_summary(data):
    return pd.DataFrame({
        "Frequency": data.count(),
        "Mean": data.mean(),
        "Median": data.median(),
        "Mode": data.mode().iloc[0],
        "Variance": data.var(),
        "Std Deviation": data.std(),
        "Skewness": data.apply(lambda x: skew(x, bias=False)),
        "Kurtosis": data.apply(lambda x: kurtosis(x, bias=False))
    })

print("PIMA – Univariate statistics")
display(univariate_summary(pima))
print("UCI – Univariate statistics")
display(univariate_summary(uci))

## 5. Bivariate analysis – Linear regression

In [ ]:
# Pima: use Glucose as one predictor and BMI as continuous response.
pima_biv = LinearRegression()
pima_biv.fit(pima[["Glucose"]], pima["BMI"])
pima_fit = pima_biv.predict(pima[["Glucose"]])
print("Pima: BMI from Glucose | slope:", pima_biv.coef_[0],
      "| intercept:", pima_biv.intercept_, "| R2:", r2_score(pima["BMI"], pima_fit))

plt.scatter(pima["Glucose"], pima["BMI"], alpha=0.4)
plt.plot(pima["Glucose"], pima_fit, color="red")
plt.xlabel("Glucose"); plt.ylabel("BMI"); plt.title("Pima Bivariate Linear Regression")
plt.show()

# UCI: use BMI to predict continuous disease progression.
uci_biv = LinearRegression()
uci_biv.fit(uci[["bmi"]], uci["DiseaseProgression"])
uci_fit = uci_biv.predict(uci[["bmi"]])
print("UCI: Progression from BMI | slope:", uci_biv.coef_[0],
      "| intercept:", uci_biv.intercept_, "| R2:", r2_score(uci["DiseaseProgression"], uci_fit))

plt.scatter(uci["bmi"], uci["DiseaseProgression"], alpha=0.4)
plt.plot(uci["bmi"], uci_fit, color="red")
plt.xlabel("BMI (standardized)"); plt.ylabel("Disease Progression")
plt.title("UCI Bivariate Linear Regression"); plt.show()

## 6. Bivariate analysis – Logistic regression

In [ ]:
# Pima Outcome is binary: 0 = non-diabetic, 1 = diabetic.
pima_log = LogisticRegression(max_iter=1000)
pima_log.fit(pima[["Glucose"]], pima["Outcome"])
pima_log_pred = pima_log.predict(pima[["Glucose"]])
print("Pima Logistic Regression (Glucose -> Outcome)")
print("Accuracy:", accuracy_score(pima["Outcome"], pima_log_pred))
print(classification_report(pima["Outcome"], pima_log_pred))

# UCI target is continuous. For a logistic demonstration, define high progression
# as values >= median (1), and low progression as values < median (0).
uci_binary = (uci["DiseaseProgression"] >= uci["DiseaseProgression"].median()).astype(int)
uci_log = LogisticRegression(max_iter=1000)
uci_log.fit(uci[["bmi"]], uci_binary)
uci_log_pred = uci_log.predict(uci[["bmi"]])
print("UCI Logistic Regression (BMI -> high/low progression)")
print("Accuracy:", accuracy_score(uci_binary, uci_log_pred))
print(classification_report(uci_binary, uci_log_pred))

## 7. Multiple regression analysis

In [ ]:
# Pima multiple linear regression: predict BMI using several measurements.
pima_features = ["Pregnancies", "Glucose", "BloodPressure", "Age"]
Xp, yp = pima[pima_features], pima["BMI"]
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    Xp, yp, test_size=0.20, random_state=42)
pima_multi = LinearRegression().fit(Xp_train, yp_train)
yp_pred = pima_multi.predict(Xp_test)
print("Pima Multiple Regression | R2:", r2_score(yp_test, yp_pred),
      "| RMSE:", np.sqrt(mean_squared_error(yp_test, yp_pred)))
print("Coefficients:", dict(zip(pima_features, pima_multi.coef_)))

# UCI multiple linear regression: predict progression using all 10 features.
uci_features = uci_raw.feature_names
Xu, yu = uci[uci_features], uci["DiseaseProgression"]
Xu_train, Xu_test, yu_train, yu_test = train_test_split(
    Xu, yu, test_size=0.20, random_state=42)
uci_multi = LinearRegression().fit(Xu_train, yu_train)
yu_pred = uci_multi.predict(Xu_test)
print("UCI Multiple Regression | R2:", r2_score(yu_test, yu_pred),
      "| RMSE:", np.sqrt(mean_squared_error(yu_test, yu_pred)))
print("Coefficients:", dict(zip(uci_features, uci_multi.coef_)))

## 8. Compare datasets

In [ ]:
comparison = pd.DataFrame({
    "Dataset": ["Pima Indians Diabetes", "UCI Diabetes"],
    "Samples": [len(pima), len(uci)],
    "Predictor Features": [8, len(uci_raw.feature_names)],
    "Target": ["Binary diabetes Outcome", "Continuous disease progression"]
})
display(comparison)

print("Pima is a classification dataset with a binary diabetes outcome.")
print("UCI Diabetes is a regression dataset with a continuous progression score.")
print("Their target definitions and feature scales differ; interpret metrics accordingly.")

## Result / Conclusion
Univariate statistics described both datasets. Bivariate linear regression examined one predictor and a continuous target; logistic regression modeled binary outcomes (the UCI progression target was converted to a median-based binary label only for demonstration). Multiple regression used several predictors. Pima and UCI differ in target meaning and should be compared with this distinction in mind.